# Crowfeather-50M-v1 — end-to-end training notebook

Trains a 50.8M-parameter dense Qwen3 model on consumer-grade hardware (Colab A100). Custom 32K Byte-Level BPE with per-digit number wrap and Fill-in-the-Middle (FIM) support. Distillation pretrain on R1 + Sonnet + NuminaMath + MetaMathQA + Opus traces.

**Pipeline**
1. Setup (Drive mount, deps, auth)
2. Precache distillation traces
3. Phase 0 — Train 32K BPE tokenizer
4. Build fresh dense Qwen3 50M init
5. Smoke test
6. Phase 1 — Pretrain @ 4K context (40K steps, FIM rate 0.5)
7. Phase 2 — CPT @ 16K context (2.5K steps)
8. Phase 3 — SFT @ 4K context (2.5K steps)
9. Quick eval / generation samples
10. Push to HF Hub

**Resumable**: every phase skips if its `final/` directory already exists. Reconnect after a session timeout and re-run from top.

**Required runtime**: A100 (80GB recommended; 40GB supported with batch adjust). T4/V100 not supported.


## 1. Setup

Mount Drive (so model checkpoints survive Colab session timeouts), install deps, and clone the repo.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/crowfeather_50m_v1'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')


In [ ]:
# Install deps. Skip if already installed (Colab caches).
import subprocess, sys

DEPS = [
    'transformers>=4.51',
    'tokenizers>=0.20',
    'datasets',
    'accelerate',
    'huggingface_hub',
    'wandb',
    'liger-kernel',  # optional, falls back gracefully
    'bitsandbytes',  # optional, for 8-bit AdamW if needed
]

for dep in DEPS:
    try:
        # Conservative: pip install -U if not present at the right version
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', dep], check=False)
    except Exception as e:
        print(f'  WARN: failed to install {dep}: {e}')

print('  done.')


In [ ]:
# Clone the repo into Colab session
REPO_URL = 'https://github.com/Crownelius/crowfeather-50m-v1.git'
REPO_DIR = '/content/crowfeather-50m-v1'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

print(f'Repo at: {REPO_DIR}')
print(os.listdir(REPO_DIR))


## 2. Authentication (HuggingFace + Weights & Biases)

**Secrets handling**: this notebook reads tokens from Colab Secrets (key icon in left sidebar). Set up before running:

1. **HF_TOKEN** — get from https://huggingface.co/settings/tokens (write-access if you plan to push)
2. **WANDB_API_KEY** — get from https://wandb.ai/authorize (read-only is fine)

If a secret is missing the cell will warn and continue without that integration (training still works; you just won't push to HF or log to wandb).

**Never paste a real token into the notebook.** Use Colab Secrets. The placeholder strings below are intentionally invalid and will not work for auth.


In [ ]:
import os
from google.colab import userdata

def _load_secret(key, fallback_placeholder):
    try:
        v = userdata.get(key)
        if v and not v.startswith('<PASTE'):
            return v
    except Exception:
        pass
    print(f'  WARN: Colab Secret {key!r} not set; using placeholder (auth will fail)')
    return fallback_placeholder

HF_TOKEN = _load_secret('HF_TOKEN', '<PASTE_YOUR_HF_TOKEN_HERE>')
WANDB_API_KEY = _load_secret('WANDB_API_KEY', '<PASTE_YOUR_WANDB_KEY_HERE>')

# Set env so child processes (subprocess.run, !command, the train scripts) can see them
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_PROJECT'] = 'crowfeather-50m'

# HF login (best-effort)
try:
    from huggingface_hub import login as hf_login
    if not HF_TOKEN.startswith('<PASTE'):
        hf_login(token=HF_TOKEN, add_to_git_credential=False)
        print('  HF login: OK')
    else:
        print('  HF login: SKIPPED (no token)')
except Exception as e:
    print(f'  HF login failed: {e}')

# wandb login (best-effort)
try:
    import wandb
    if not WANDB_API_KEY.startswith('<PASTE'):
        wandb.login(key=WANDB_API_KEY, relogin=False)
        print('  wandb login: OK')
    else:
        print('  wandb login: SKIPPED (no key)')
except Exception as e:
    print(f'  wandb login failed: {e}')


## 3. Configuration

Auto-detect GPU type and set batch sizes accordingly. All paths and constants live in this cell — change here only.


In [ ]:
import torch

# GPU detection
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
gpu_total = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
IS_80GB = gpu_total > 60
print(f'GPU: {gpu_name} ({gpu_total:.0f} GB)  IS_80GB={IS_80GB}')

# Paths
OUTPUT_DIR = f'{DRIVE_ROOT}/output'
DISTILL_CACHE = f'{OUTPUT_DIR}/distill_data'
TOKENIZER_DIR = f'{OUTPUT_DIR}/tokenizer'
INIT_DIR = f'{OUTPUT_DIR}/init'
PHASE1_DIR = f'{OUTPUT_DIR}/phase1'
PHASE2_DIR = f'{OUTPUT_DIR}/phase2'
PHASE3_DIR = f'{OUTPUT_DIR}/phase3'

for d in [OUTPUT_DIR, DISTILL_CACHE]:
    os.makedirs(d, exist_ok=True)

# Phase 1 batch + accum (effective batch = 8 either way)
B1   = 4 if IS_80GB else 2
ACC1 = 2 if IS_80GB else 4

# Phase 2 batch + accum (effective batch = 4 either way)
B2   = 2 if IS_80GB else 1
ACC2 = 2 if IS_80GB else 4

# Phase 3 batch + accum (effective batch = 8 either way)
B3   = 4 if IS_80GB else 2
ACC3 = 2 if IS_80GB else 4

print(f'Phase 1: B={B1}  accum={ACC1}  eff={B1*ACC1}  T=4096  steps=40000')
print(f'Phase 2: B={B2}  accum={ACC2}  eff={B2*ACC2}  T=16384 steps=2500')
print(f'Phase 3: B={B3}  accum={ACC3}  eff={B3*ACC3}  T=4096  steps=2500')


## 4. Precache distillation traces

Stream traces from HuggingFace into per-domain JSONL files on Drive. Cached: subsequent runs skip if the files already exist with reasonable size.

Default budget: 8 GB total (3 GB math + 3 GB code + 2 GB lang). Adjust `--budget-mb` if you have less Drive space.


In [ ]:
# Skip if already done
math_jsonl = f'{DISTILL_CACHE}/math.jsonl'
lang_jsonl = f'{DISTILL_CACHE}/lang.jsonl'
code_jsonl = f'{DISTILL_CACHE}/code.jsonl'

need_precache = any(
    not os.path.exists(p) or os.path.getsize(p) < 1_000_000
    for p in [math_jsonl, lang_jsonl, code_jsonl]
)

if need_precache:
    print('Running precache...')
    !python {REPO_DIR}/scripts/precache_distill.py \
        --target-dir {DISTILL_CACHE} \
        --drive-cache {DRIVE_ROOT}/distill_data \
        --budget-mb 8000
else:
    print('Precache cache already populated; skipping.')
    for p in [math_jsonl, lang_jsonl, code_jsonl]:
        sz_mb = os.path.getsize(p) / 1e6
        print(f'  {os.path.basename(p):20s} {sz_mb:8.1f} MB')


## 5. Phase 0 — Train 32K Byte-Level BPE tokenizer

Trains on the precache corpus (per-digit-wrapped before BPE training). Special tokens: pad, bos, eos, unk, FIM (4), chat roles (4), think (2), tool tags (4) = 18 reserved IDs.

~30 minutes CPU.


In [ ]:
tokenizer_json = f'{TOKENIZER_DIR}/tokenizer.json'
if os.path.exists(tokenizer_json):
    print(f'Tokenizer already trained at {TOKENIZER_DIR}; skipping.')
else:
    !python {REPO_DIR}/scripts/train_bpe.py \
        --cache-dir {DISTILL_CACHE} \
        --output-dir {TOKENIZER_DIR} \
        --vocab-size 32768 \
        --max-corpus-chars 2000000000

# Quick verify
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(TOKENIZER_DIR, use_fast=True)
print(f'  vocab_size: {len(tok)}')
print(f'  pad_id={tok.pad_token_id}  bos_id={tok.bos_token_id}  eos_id={tok.eos_token_id}')
print(f'  fim_prefix={tok.convert_tokens_to_ids(chr(60)+chr(124)+"fim_prefix"+chr(124)+chr(62))}')


## 6. Build fresh dense Qwen3 50M init

Random init from `Qwen3Config(hidden=448, layers=12, q=8, kv=4, head_dim=56, intermediate=1792, max_pos=16384, vocab=tokenizer.size)`.

Total ~50.8M parameters with tied embeddings.


In [ ]:
init_safetensors = f'{INIT_DIR}/model.safetensors'
init_index = f'{INIT_DIR}/pytorch_model.bin.index.json'
if os.path.exists(init_safetensors) or os.path.exists(init_index):
    print(f'Init already built at {INIT_DIR}; skipping.')
else:
    !python {REPO_DIR}/scripts/build_init.py \
        --tokenizer-dir {TOKENIZER_DIR} \
        --output-dir {INIT_DIR}


## 7. Smoke test

Tokenize a string, run a forward pass, verify shapes and that the loss is finite. If anything is wrong, this catches it in 30 seconds rather than 4 hours into Phase 1.


In [ ]:
import torch
from transformers import AutoTokenizer, Qwen3ForCausalLM

tok = AutoTokenizer.from_pretrained(INIT_DIR, use_fast=True)
model = Qwen3ForCausalLM.from_pretrained(INIT_DIR, torch_dtype=torch.bfloat16).cuda()
model.eval()

n_total = sum(p.numel() for p in model.parameters())
print(f'Loaded model: {n_total/1e6:.2f}M params')

txt = "The quick brown fox jumps over 1 2 3 4 5 lazy dogs."
ids = tok.encode(txt, return_tensors='pt').cuda()
print(f'  input_ids shape: {tuple(ids.shape)}')

with torch.no_grad():
    out = model(input_ids=ids, labels=ids)
print(f'  logits shape: {tuple(out.logits.shape)}')
print(f'  loss: {out.loss.item():.4f}  (expected ~10.5 for random-init 32K vocab)')
assert torch.isfinite(out.loss), 'Loss is NaN/Inf -- abort'
print('  smoke test: PASS')
del model, out, ids
torch.cuda.empty_cache()


## 8. Phase 1 — Pretrain @ 4K context (FIM rate 0.5)

The big one. 40K steps, ~4-5 hours on A100 80GB. Will checkpoint every 2,500 steps to Drive. If Colab disconnects, re-run this cell — it auto-resumes from the latest step.


In [ ]:
phase1_final = f'{PHASE1_DIR}/final'
if os.path.exists(phase1_final) and os.path.exists(f'{phase1_final}/model.safetensors'):
    print(f'Phase 1 already complete at {phase1_final}; skipping.')
else:
    # Resume from latest step ckpt if available, else from init
    import glob
    step_ckpts = sorted(glob.glob(f'{PHASE1_DIR}/step_*'))
    resume_from = step_ckpts[-1] if step_ckpts else INIT_DIR
    print(f'Phase 1 resume from: {resume_from}')

    !python {REPO_DIR}/scripts/train_dense.py \
        --phase phase1 \
        --resume {resume_from} \
        --output {PHASE1_DIR} \
        --cache-dir {DISTILL_CACHE} \
        --steps 40000 \
        --batch-size {B1} --grad-accum {ACC1} \
        --seq-len 4096 \
        --peak-lr 3e-4 --min-lr 3e-5 \
        --warmup-frac 0.015 --decay-frac 0.20 \
        --z-loss 1e-4 \
        --fim-rate 0.5 \
        --ckpt-every 2500


## 9. Phase 2 — Continued Pretrain @ 16K context

2,500 steps at 16K context, no FIM (CPT preserves long-document L-to-R structure). ~1.5 hours.


In [ ]:
phase2_final = f'{PHASE2_DIR}/final'
if os.path.exists(phase2_final) and os.path.exists(f'{phase2_final}/model.safetensors'):
    print(f'Phase 2 already complete at {phase2_final}; skipping.')
else:
    import glob
    step_ckpts = sorted(glob.glob(f'{PHASE2_DIR}/step_*'))
    resume_from = step_ckpts[-1] if step_ckpts else f'{PHASE1_DIR}/final'
    print(f'Phase 2 resume from: {resume_from}')

    !python {REPO_DIR}/scripts/train_dense.py \
        --phase cpt_16k \
        --resume {resume_from} \
        --output {PHASE2_DIR} \
        --cache-dir {DISTILL_CACHE} \
        --steps 2500 \
        --batch-size {B2} --grad-accum {ACC2} \
        --seq-len 16384 \
        --peak-lr 6e-5 --min-lr 6e-6 \
        --warmup-frac 0.05 --decay-frac 0.30 \
        --z-loss 1e-4 \
        --fim-rate 0.0 \
        --ckpt-every 500


## 10. Phase 3 — SFT @ 4K context

Final phase. 2,500 steps, no FIM, conservative LR. ~20 minutes.


In [ ]:
phase3_final = f'{PHASE3_DIR}/final'
if os.path.exists(phase3_final) and os.path.exists(f'{phase3_final}/model.safetensors'):
    print(f'Phase 3 already complete at {phase3_final}; skipping.')
else:
    import glob
    step_ckpts = sorted(glob.glob(f'{PHASE3_DIR}/step_*'))
    if step_ckpts:
        resume_from = step_ckpts[-1]
    elif os.path.exists(f'{PHASE2_DIR}/final'):
        resume_from = f'{PHASE2_DIR}/final'
    else:
        resume_from = f'{PHASE1_DIR}/final'
    print(f'Phase 3 resume from: {resume_from}')

    !python {REPO_DIR}/scripts/train_dense.py \
        --phase sft_4k \
        --resume {resume_from} \
        --output {PHASE3_DIR} \
        --cache-dir {DISTILL_CACHE} \
        --steps 2500 \
        --batch-size {B3} --grad-accum {ACC3} \
        --seq-len 4096 \
        --peak-lr 4e-5 --min-lr 4e-6 \
        --warmup-frac 0.05 --decay-frac 0.30 \
        --z-loss 1e-4 \
        --fim-rate 0.0 \
        --ckpt-every 500


## 11. Quick eval — generation samples

Load the final SFT checkpoint and generate from a few prompts. Tests both standard L-to-R and FIM infilling.


In [ ]:
import torch
from transformers import AutoTokenizer, Qwen3ForCausalLM

FINAL_DIR = f'{PHASE3_DIR}/final'
tok = AutoTokenizer.from_pretrained(FINAL_DIR, use_fast=True)
model = Qwen3ForCausalLM.from_pretrained(FINAL_DIR, torch_dtype=torch.bfloat16).cuda()
model.eval()

def gen(prompt, max_new=80, temp=0.7):
    ids = tok.encode(prompt, return_tensors='pt').cuda()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new,
                             do_sample=True, temperature=temp, top_p=0.9,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=False)

# Standard L-to-R prompts
print('=== L-to-R generation ===')
for p in [
    'The capital of France is',
    'Question: What is 2 + 2?\nAnswer:',
    'def fibonacci(n):',
]:
    print(f'>>> {p!r}')
    print(f'    {gen(p)!r}')
    print()

# FIM infilling prompt
print('=== FIM infilling ===')
fim_prompt = '<|fim_prefix|>def add(a, b):<|fim_suffix|>    return result<|fim_middle|>'
print(f'>>> prefix="def add(a, b):"  suffix="    return result"')
print(f'    middle: {gen(fim_prompt)!r}')

del model
torch.cuda.empty_cache()


## 12. Push to HuggingFace Hub

Pushes the final SFT checkpoint to HF Hub. Requires HF_TOKEN with write access (set in Colab Secrets).

Repo target: `CompactAI-O/Crowfeather-50M-v1`. Change `HF_REPO` below if pushing somewhere else.


In [ ]:
HF_REPO = 'CompactAI-O/Crowfeather-50M-v1'

if HF_TOKEN.startswith('<PASTE'):
    print('HF_TOKEN not set; skipping push.')
else:
    from huggingface_hub import HfApi, create_repo
    api = HfApi()
    try:
        create_repo(HF_REPO, private=False, exist_ok=True, token=HF_TOKEN)
    except Exception as e:
        print(f'  create_repo: {e}')

    print(f'Uploading {FINAL_DIR} to {HF_REPO} ...')
    api.upload_folder(
        folder_path=FINAL_DIR,
        repo_id=HF_REPO,
        commit_message='Crowfeather-50M-v1 SFT final',
        token=HF_TOKEN,
    )
    print(f'  done: https://huggingface.co/{HF_REPO}')


## 13. (Optional) GGUF export

Once `llama.cpp` supports the architecture (Qwen3 already supported), convert via:

```bash
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp && pip install -r requirements.txt
python convert_hf_to_gguf.py /path/to/Crowfeather-50M-v1/final --outfile crowfeather-50m-v1.gguf --outtype f16
./build/bin/llama-quantize crowfeather-50m-v1.gguf crowfeather-50m-v1-Q4_K_M.gguf Q4_K_M
```

For 50M models, Q5_K_M is the sweet spot (negligible quality loss, ~25% smaller than f16).
